# Test polytopes' volume computation — CNN

Analogous to `test_volumes.ipynb` but for `FashionCNN_Small`.

Uses `build_cnn_correct_polytope` / `build_cnn_all_polytopes` from
`src/optim/build_polytopes_cnn.py`, and `estimate_polytope_width` from
`src/optim/compute_volumes.py`.

`estimate_polytope_width` internally calls `_prep_lp` which:
- **prunes zero-norm rows** (trivial constraints from saturated MaxPool neurons)
- **adds ε = 1e-6 slack** to absorb floating-point violations at x₀

Both fixes are critical for CNN LP feasibility (without them HiGHS declares
infeasible even though x₀ is inside the polytope).

**LP scalability note**: use a small `nb_directions` (3–5) to get a timing
estimate before scaling up on Jean Zay.

## Libraries

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(ROOT, "data")
sys.path.append(ROOT)

print("Added to path:", ROOT)

import torch

from src.models.networks import FashionCNN_Small
from src.quantization.quantize import quantize_model
from src.optim.build_polytopes_cnn import build_cnn_correct_polytope, build_cnn_all_polytopes
from src.optim.compute_volumes import estimate_polytope_width

Added to path: /Users/jeremiecabessa/Desktop/ROOT/Articles/Conference_Papers/2025_With_Jiri/code/ErrorVolumePolytopes


In [2]:
torch.manual_seed(42)

## CNN (FashionMNIST)

In [3]:
# -------------------- #
# Load models and data #
# -------------------- #

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
print("Device:", device)

# Full-precision CNN
model = FashionCNN_Small()
model.load_state_dict(
    torch.load(os.path.join(ROOT, "checkpoints", "fashion_cnn_best.pth"),
               weights_only=True, map_location=device)
)
model.to(device).eval()

# Quantized model (bits=4 for the main test)
bits = 4
qmodel = quantize_model(model, bits=bits)
qmodel.to(device).eval()

# Dataset (correctly-classified samples for the CNN)
dataset = torch.load(
    DATA_PATH + "/fashionMNIST_correct_cnn.pt",
    weights_only=False
)

x_0, c = dataset[13]
print(f"x_0 range: [{x_0.min().item():.2f}, {x_0.max().item():.2f}]")  # check bounds
x_0  = x_0.to(device)          # (1, 28, 28)
x_in = x_0.unsqueeze(0)        # (1, 1, 28, 28) — batch dim for the CNN
print(f"x_in shape: {x_in.shape}  |  label: {c}")
print("Models and dataset have been loaded.")

Device: mps
x_0 range: [-1.00, 0.99]
x_in shape: torch.Size([1, 1, 28, 28])  |  label: 5
Models and dataset have been loaded.


## Correct polytope — LP timing

Build the correct polytope (model-only, no qmodel) and estimate its mean width
with a small number of directions to measure per-LP cost.

`_prep_lp` is called automatically inside `estimate_polytope_width`:
it prunes zero-norm rows and adds ε slack before passing to HiGHS.

In [4]:
%%time

print("*** Correct polytope (model-only) ***\n")

# Keep small — each LP can take several minutes for a CNN.
# Increase to 50–200 on Jean Zay.
nb_directions = 3

# bounds apply to all 784 input variables
bounds = [(-1., 1.)]

A_correct, b_correct = build_cnn_correct_polytope(model, x_in, int(c))
print(f"A_correct shape: {tuple(A_correct.shape)}")

results_correct = estimate_polytope_width(
    A_correct, b_correct, bounds,
    n_directions=nb_directions,
    verbose=True,
)
results_correct

*** Correct polytope (model-only) ***

A_correct shape: (20685, 784)
Direction 0: w_correct=1.3499
Direction 1: w_correct=1.4827
Direction 2: w_correct=1.4805
CPU times: user 32.2 s, sys: 736 ms, total: 32.9 s
Wall time: 37.1 s


{'mean_width_correct': 1.4376934015479819,
 'std_width_correct': 0.0620810833942755,
 'n_directions_used': 3}

## b-approximated polytope — paired LP (same directions)

Estimate mean width of `A_both` (model + qmodel constraints) using the **same**
random directions as above (paired mode).  This gives the error estimate:

    error = 1 - mean_width_both / mean_width_correct

In [5]:
%%time

print(f"*** b-approximated polytope (bits={bits}) — paired estimation ***\n")

# build_cnn_all_polytopes reuses model constraints internally (computed once).
# We pass only the one qmodel we care about here.
_, _, polytopes_dict = build_cnn_all_polytopes(model, {bits: qmodel}, x_in, int(c))
A_both, b_both = polytopes_dict[bits]
print(f"A_both shape: {tuple(A_both.shape)}")

results_both = estimate_polytope_width(
    A_correct, b_correct, bounds,
    A_both=A_both, b_both=b_both,
    n_directions=nb_directions,
    verbose=True,
)
results_both

*** b-approximated polytope (bits=4) — paired estimation ***

A_both shape: (41370, 784)
Direction 0: w_correct=1.3228, w_both=0.5777
Direction 1: w_correct=1.4166, w_both=0.6401
Direction 2: w_correct=1.3098, w_both=0.5832
CPU times: user 2min 32s, sys: 2.57 s, total: 2min 35s
Wall time: 2min 36s


{'mean_width_correct': 1.3497586155638122,
 'std_width_correct': 0.04757989866158496,
 'n_directions_used': 3,
 'mean_width_both': 0.6003570219649221,
 'std_width_both': 0.02818468087729982,
 'error': 0.5552115652070537}

## Sanity check — model and qmodel predictions

In [6]:
with torch.no_grad():
    pred_model  = torch.argmax(model(x_in)).item()
    pred_qmodel = torch.argmax(qmodel(x_in)).item()
print(f"model  prediction : {pred_model}")
print(f"qmodel prediction : {pred_qmodel}")
print(f"true label        : {int(c)}")

model  prediction : 5
qmodel prediction : 5
true label        : 5
